In [1]:
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 9.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 37.9 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
import torch
import os
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig 
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

In [3]:
# 1. Authenticate with Hugging Face 
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

os.environ["WANDB_DISABLED"] = "true"

In [4]:
# 2. Load the ENTIRE Dataset (No more [:5000] slicer)
dataset = load_dataset("bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset", split="train")

def format_instruction(example):
    sys_prompt = "You are a polite and helpful e-commerce customer support AI."
    text = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{sys_prompt}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{example['instruction']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n{example['response']}<|eot_id|>"
    )
    return {"text": text}

formatted_dataset = dataset.map(format_instruction)

README.md: 0.00B [00:00, ?B/s]

bitext-retail-ecommerce-llm-chatbot-trai(…):   0%|          | 0.00/42.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44884 [00:00<?, ? examples/s]

Map:   0%|          | 0/44884 [00:00<?, ? examples/s]

In [5]:
# 3. Native 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16   
)


In [6]:
# 4. Load Model and Tokenizer (Forcing Float16 for T4 GPU)
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16  
)

# Force the model config to float16 to stop LoRA adapters from spawning in BFloat16
model.config.torch_dtype = torch.float16

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [7]:
# 5. Prepare Native PEFT / LoRA
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [8]:
# 6. SFTConfig (Full Epoch Run with Checkpointing)
training_args = SFTConfig(
    output_dir="./ecommerce-bot-results",
    per_device_train_batch_size=1,        
    gradient_accumulation_steps=8,        
    optim="paged_adamw_8bit",             
    
    num_train_epochs=1,           # Trains on the full 44.9k dataset once
    save_strategy="steps",
    save_steps=100,               # Saves a backup checkpoint every 100 steps
    save_total_limit=2,           # Keeps disk space clear by deleting older backups
    
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,             
    bf16=False,            
    warmup_ratio=0.03,
    gradient_checkpointing=True,
    dataset_text_field="text",    
    max_length=512,           
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
# 7. Initialize Trainer
trainer = SFTTrainer(
    model=model,                  
    train_dataset=formatted_dataset,
    peft_config=peft_config,      
    processing_class=tokenizer,   
    args=training_args,
)


Adding EOS to train dataset:   0%|          | 0/44884 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/44884 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/44884 [00:00<?, ? examples/s]

In [10]:
# 8. THE ULTIMATE FAILSAFE (Protects Float32 adapters & converts rogue BFloat16)
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
    elif param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

In [11]:
# 9. IGNITION
print("🚀 IGNITION: Starting the full dataset fine-tuning process...")
torch.cuda.empty_cache() 
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


🚀 IGNITION: Starting the full dataset fine-tuning process...


Step,Training Loss
10,2.042623
20,1.719258
30,1.236577
40,1.084073
50,0.944849
60,0.861371
70,0.795861
80,0.761997
90,0.723305
100,0.668827


KeyboardInterrupt: 

In [ ]:
# 10. Save the adapter
trainer.model.save_pretrained("ecommerce-bot-adapter")
tokenizer.save_pretrained("ecommerce-bot-adapter")
print("Full training complete and model saved!")

In [12]:
import shutil

# Zipping with absolute Kaggle paths
shutil.make_archive(
    base_name="/kaggle/working/ecommerce_model_ready", # Where to save the zip file
    format="zip", 
    root_dir="/kaggle/working/ecommerce-bot-results"   # The exact path to the folder to zip
)

print("✅ Model successfully zipped! Look for 'ecommerce_model_ready.zip' in your output folder.")

✅ Model successfully zipped! Look for 'ecommerce_model_ready.zip' in your output folder.


In [6]:
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import re

# 1. Load the Base Model and Tokenizer (in 4-bit to save memory)
base_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
adapter_path = "/kaggle/working/ecommerce-bot-results/checkpoint-800" # Change to your checkpoint if needed

print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16
)

# 2. Snap on your fine-tuned LoRA weights
print("Applying fine-tuned adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)

# 3. The Core Chat Logic
def respond(message, history):
    sys_prompt = "You are a polite and helpful e-commerce customer support AI."
    
    # Start with the system prompt
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{sys_prompt}<|eot_id|>"
    
    # Loop through the history to give the bot a memory of the conversation
    for user_msg, bot_msg in history:
        prompt += f"<|start_header_id|>user<|end_header_id|>\n\n{user_msg}<|eot_id|>"
        prompt += f"<|start_header_id|>assistant<|end_header_id|>\n\n{bot_msg}<|eot_id|>"
        
    # Add the brand new message from the user
    prompt += f"<|start_header_id|>user<|end_header_id|>\n\n{message}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs, 
        max_new_tokens=150, 
        temperature=0.3, 
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Decode only the newly generated text
    response_token_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(response_token_ids, skip_special_tokens=True)
    clean_response = re.sub(r'\{\{(.*?)\}\}', r'\1', response)
    
    return clean_response.strip()

# 4. Build and Launch the Visual Interface
print("Launching UI...")

demo = gr.ChatInterface(
    fn=respond,
    title="🛒 E-Commerce Support Bot",
    description="I am your fine-tuned Llama 3 assistant. Ask me about your orders, returns, or shipping!",
    theme="soft"
)

# share=True creates a temporary public link you can open on your phone!
demo.launch(share=True)

Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Applying fine-tuned adapter...
Launching UI...


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://292d0794dd573f004d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
